# Session 8 — Streaming Responses

Streaming is one of the key differences between a static notebook demo and a responsive application. This notebook shows the event pattern we will later reuse in Gradio and Streamlit.

## Learning Goals

- understand why streaming improves app experience
- use the Responses API with `stream=True`
- process text deltas from the event stream
- connect the same pattern to later UI examples
- compare streaming patterns across OpenAI, Ollama, and GitHub Models


In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_ORG_ID = os.getenv('OPENAI_ORG_ID')
OPENAI_PROJECT_ID = os.getenv('OPENAI_PROJECT_ID')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
print('OpenAI key present:', bool(OPENAI_API_KEY))
print('OpenAI org ID present:', bool(OPENAI_ORG_ID))
print('OpenAI project ID present:', bool(OPENAI_PROJECT_ID))

print('GitHub token present:', bool(GITHUB_TOKEN))

OpenAI key present: True
OpenAI org ID present: True
OpenAI project ID present: True
GitHub token present: True


## Non-Streaming Baseline

Start with a normal Response API call so we can compare it to the streaming version.

In [2]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response = client.responses.create(
    model='gpt-4o-mini',
    instructions='You are a concise teaching assistant.',
    input='Explain in two short sentences why streaming is useful in chat apps.'
)

display(Markdown(response.output_text))

Streaming in chat apps allows real-time communication, making conversations more dynamic and engaging. It also facilitates instant sharing of multimedia content, enhancing user interaction and connectivity.

## Streaming with the Responses API

Now we ask the same style of question, but receive output incrementally as events arrive.

In [3]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response_stream = client.responses.create(
    model='gpt-4o-mini',
    instructions='You are a concise teaching assistant.',
    input='Explain in two short sentences why streaming is useful in chat apps.',
    stream=True,
)

streamed_text = ''
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for event in response_stream:
    if event.type == 'response.output_text.delta':
        streamed_text += event.delta
        handle.update(Markdown(streamed_text))

Streaming in chat apps allows for real-time communication, enabling users to receive updates and messages instantly. This enhances engagement and facilitates dynamic conversations, making interactions more interactive and fluid.

## Streaming with Ollama

Ollama can expose an OpenAI-compatible Responses API. That means the same event pattern can work against a local model server, as long as you already have a compatible model running locally.


In [6]:
# Optional: Ollama streaming with the OpenAI SDK and the Responses API
# This requires Ollama running locally and a model already pulled,
# for example: `ollama pull qwen3:8b`.

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
OLLAMA_CHAT_MODEL = os.getenv('OLLAMA_CHAT_MODEL', 'llama3.2:latest')

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
ollama_stream = ollama_client.responses.create(
    model=OLLAMA_CHAT_MODEL,
    input='Explain in two short sentences why streaming is useful in chat apps.',
    stream=True,
)

ollama_text = ''
handle = display(Markdown('_Ollama streaming response will appear here..._'), display_id=True)

for event in ollama_stream:
    if event.type == 'response.output_text.delta':
        ollama_text += event.delta
        handle.update(Markdown(ollama_text))


Streaming allows for real-time audio and video transmission, enabling seamless voice or video conversations across vast distances. It also reduces latency and packet loss, resulting in a more stable and immersive communication experience.

## Streaming with GitHub Models

GitHub Models supports streaming chat completions on its inference endpoint. In this notebook, that means the streaming loop is similar, but the API surface is `chat.completions` rather than `responses`. Some streamed chunks may not contain text, so the loop below skips empty chunks before reading `choices[0]`.


In [11]:
# Optional: GitHub Models streaming with chat completions
# GitHub Models supports chat completions streaming on its inference endpoint,
# but not the Responses API used in the main OpenAI examples above.
# Some streamed chunks may be housekeeping events with no choices, so guard
# before reading `choices[0]`.

github_client = OpenAI(
    base_url='https://models.github.ai/inference',
    api_key=GITHUB_TOKEN,
)

github_model = os.getenv('GITHUB_CHAT_MODEL', 'openai/gpt-4.1')

github_stream = github_client.chat.completions.create(
    model=github_model,
    messages=[
        {'role': 'system', 'content': 'You are a concise teaching assistant.'},
        {'role': 'user', 'content': 'Explain in two short sentences why streaming is useful in chat apps.'},
    ],
    stream=True,
    max_tokens=120,
    extra_headers={
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2026-03-10',
    },
)

github_text = ''
handle = display(Markdown('_GitHub Models streaming response will appear here..._'), display_id=True)

for chunk in github_stream:
    if not chunk.choices:
        continue

    delta = chunk.choices[0].delta.content or ''
    if not delta:
        continue

    github_text += delta
    handle.update(Markdown(github_text))

display(Markdown(github_text or '_No text delta was returned by the stream._'))


Streaming allows chat apps to deliver messages instantly, enabling real-time communication. It also reduces latency and resource usage by sending data gradually, rather than all at once.

Streaming allows chat apps to deliver messages instantly, enabling real-time communication. It also reduces latency and resource usage by sending data gradually, rather than all at once.

The key teaching point is that streaming is the common idea, but the exact client call depends on the provider. OpenAI and Ollama can both demonstrate the Responses API pattern here, while GitHub Models currently fits the chat completions pattern instead.


## Turn the Event Loop into a Generator

UI frameworks often want a generator or iterable. That makes streaming a natural fit.

In [12]:
def stream_response(prompt: str, model: str = 'gpt-4o-mini'):
    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
    stream = client.responses.create(
        model=model,
        instructions='You are a concise teaching assistant.',
        input=prompt,
        stream=True,
    )

    for event in stream:
        if event.type == 'response.output_text.delta':
            yield event.delta

In [13]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

collected = []
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for piece in stream_response('Give one short paragraph on why streamed output feels faster to users.'):
    collected.append(piece)
    handle.update(Markdown(''.join(collected)))

full_text = ''.join(collected)

display(Markdown('### Final combined text'))
display(Markdown(full_text))

Streamed output feels faster to users because it allows them to receive and interact with data progressively, rather than waiting for the entire batch to load. This incremental delivery creates a sense of immediacy, as users can start engaging with content without delay. The perception of speed is enhanced as users see results appearing in real-time, which keeps their attention and fosters a smoother overall experience.

### Final combined text

Streamed output feels faster to users because it allows them to receive and interact with data progressively, rather than waiting for the entire batch to load. This incremental delivery creates a sense of immediacy, as users can start engaging with content without delay. The perception of speed is enhanced as users see results appearing in real-time, which keeps their attention and fosters a smoother overall experience.

## Why This Matters for the Rest of Session 8

- The Gradio demo can consume a generator like `stream_response(...)`.
- The Streamlit app can pass the same generator to `st.write_stream(...)`.
- Streaming is not a separate concept from app building; it is one of the main reasons the app feels interactive.